# USMAN AND UMAR — Colab Backend with Localtunnel
This notebook runs the OpenSwarm multi-agent executor in Google Colab and exposes it through Localtunnel. `/execute` remains protected by `Authorization: Bearer <APP_CLIENT_TOKEN>`.
## One-time setup
1. In Colab, add `APP_CLIENT_TOKEN` in the **Secrets** panel and allow notebook access. Provider secrets are optional for scraping; add `NVIDIA_API_KEY` or `OPENROUTER_API_KEY` only if you plan to use OpenSwarm. NVIDIA takes priority if both are present.
2. NVIDIA is pinned to `nvidia/nemotron-3.5-bolt-e4b`. Optionally add `OPENROUTER_MODEL` (default `openrouter/free`) as a Colab Secret.
3. Run the fast setup, start FastAPI, then start Localtunnel. The stable requested URL is `https://usman-umar.loca.lt`; copy that base URL into the Android app's **Settings → Cloud Servers**. The app adds `/execute` automatically.
4. **Open the URL in your browser first to bypass the Localtunnel warning page.** Keep the Colab runtime and Localtunnel cell alive while using cloud tasks.

Localtunnel subdomains are requested rather than guaranteed reservations; if `usman-umar` is unavailable, change `LOCALTUNNEL_SUBDOMAIN` in the startup cell and update the app URL.

Browser tasks must include a public `http(s)` URL in the prompt. The Playwright helper extracts page text and can click a requested link by name. Do not include secrets or private information. Full server tracebacks are logged to `/tmp/usman-fastapi.log`; the public API returns a credential-safe error summary.


## 1. Install Localtunnel
Install the Localtunnel client and request the stable `usman-umar` subdomain.


In [ ]:
!npm install -g localtunnel
!lt --version


## 2. Fast backend startup and isolated AI environment
The setup cell creates `/content/venv`. Cell A installs only startup dependencies into that environment. The FastAPI process also uses that environment, so Colab's pre-installed packages are not modified.

After FastAPI is online, start Localtunnel in the next section. Then run Cell B. Its heavy packages install into the same virtual environment while the detached server remains available.


In [ ]:
# Colab runtimes can lack the bundled venv module dependencies; virtualenv is more robust.
!pip install -q virtualenv
import os
import shutil
import subprocess
import sys
from pathlib import Path
REPO_URL = "https://github.com/saedhawwa23-rgb/usman-and-umar-app.git"
BRANCH = "feature/hybrid-task-orchestrator"
PROJECT_DIR = Path("/content/usman-and-umar-app")
VENV_DIR = Path("/content/venv")
VENV_PYTHON = VENV_DIR / "bin" / "python"
VENV_PIP = VENV_DIR / "bin" / "pip"
if not (PROJECT_DIR / ".git").exists():
    if PROJECT_DIR.exists():
        raise RuntimeError(
            f"{PROJECT_DIR} exists but is not a Git checkout. Move it aside and rerun this cell."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )
else:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=PROJECT_DIR,
        check=True,
    )
if not VENV_PYTHON.exists():
    try:
        subprocess.run(["virtualenv", str(VENV_DIR)], check=True)
    except subprocess.CalledProcessError:
        print("virtualenv creation failed; deleting the partial environment and retrying once...")
        shutil.rmtree(VENV_DIR, ignore_errors=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "virtualenv"], check=True)
        subprocess.run(["virtualenv", str(VENV_DIR)], check=True)
BACKEND_DIR = PROJECT_DIR / "cloud-backend"
if not (BACKEND_DIR / "main.py").is_file() or not (BACKEND_DIR / "browser_scraper.py").is_file():
    raise RuntimeError("The expected FastAPI or Playwright scraper source file is missing from the checkout.")
PLAYWRIGHT_BROWSERS_PATH = "/content/.cache/ms-playwright"
os.environ["PLAYWRIGHT_BROWSERS_PATH"] = PLAYWRIGHT_BROWSERS_PATH
os.makedirs(PLAYWRIGHT_BROWSERS_PATH, exist_ok=True)
print(f"Backend checkout ready at {PROJECT_DIR}")
print(f"Isolated Python environment ready at {VENV_DIR}")


In [ ]:
# Cell A — fast startup dependencies, isolated from Colab's base Python.
!/content/venv/bin/pip install fastapi uvicorn requests python-multipart
print("FastAPI startup dependencies are ready in /content/venv.")


## 3. Read Colab Secrets (values are never printed)
Run this after Cell A. Required: `APP_CLIENT_TOKEN`. Inference keys are optional unless using OpenSwarm.


In [ ]:
import os
from google.colab import userdata

def read_secret(name, *, required=False):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    value = (value or "").strip()
    if required and not value:
        raise RuntimeError(
            f"Missing Colab Secret {name}. Add it in the Secrets panel and enable notebook access."
        )
    return value

APP_CLIENT_TOKEN = read_secret("APP_CLIENT_TOKEN", required=True)
NVIDIA_API_KEY = read_secret("NVIDIA_API_KEY")
OPENROUTER_API_KEY = read_secret("OPENROUTER_API_KEY")
OPENROUTER_MODEL = read_secret("OPENROUTER_MODEL") or os.environ.get("OPENROUTER_MODEL", "").strip() or "openrouter/free"

os.environ["APP_CLIENT_TOKEN"] = APP_CLIENT_TOKEN
if NVIDIA_API_KEY:
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
if OPENROUTER_API_KEY:
    os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
os.environ["OPENROUTER_MODEL"] = OPENROUTER_MODEL

selected_provider = "NVIDIA" if NVIDIA_API_KEY else ("OpenRouter" if OPENROUTER_API_KEY else "none (scraping only)")
print(f"Secrets loaded; selected inference provider: {selected_provider}. Key values are hidden.")


## 4. Start the FastAPI server on port 7860
This starts immediately because the heavy AI/browser modules are lazy-loaded only when `/execute` receives a task.


In [ ]:
import os
import subprocess
import time
import urllib.request
from pathlib import Path
API_BASE = "http://127.0.0.1:7860"
def local_health_ok():
    try:
        with urllib.request.urlopen(f"{API_BASE}/health", timeout=2) as response:
            return response.status == 200
    except Exception:
        return False
if local_health_ok():
    print("FastAPI is already healthy on port 7860.")
else:
    server_log_path = Path("/tmp/usman-fastapi.log")
    server_log = open(server_log_path, "a", encoding="utf-8")
    server_process = subprocess.Popen(
        ["/content/venv/bin/python", "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"],
        cwd=BACKEND_DIR,
        env=os.environ.copy(),
        stdout=server_log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    for _ in range(30):
        if local_health_ok():
            break
        if server_process.poll() is not None:
            break
        time.sleep(1)
    if not local_health_ok():
        server_log.flush()
        server_log.close()
        log_text = server_log_path.read_text(encoding="utf-8", errors="replace")[-3000:]
        raise RuntimeError(f"FastAPI did not become healthy. Recent server log:\n{log_text}")
    print("FastAPI health check passed using /content/venv/bin/python.")


## 5. Start Localtunnel and copy the stable URL
Run this after the FastAPI health check succeeds and keep the cell running. The command requests `https://usman-umar.loca.lt`. **Open the URL in your browser first to bypass the Localtunnel warning page.** Then copy the base URL into the Android app's Cloud Servers settings; `/execute` is appended automatically.


In [ ]:
import os
import subprocess
# Fixed startup command: !lt --port 7860 --subdomain usman-umar
LOCALTUNNEL_SUBDOMAIN = os.environ.get("LOCALTUNNEL_SUBDOMAIN", "usman-umar")
print(f"Starting Localtunnel at https://{LOCALTUNNEL_SUBDOMAIN}.loca.lt ...")
print("Open the URL in your browser first to bypass the Localtunnel warning page.")
localtunnel_process = subprocess.Popen(
    ["lt", "--port", "7860", "--subdomain", LOCALTUNNEL_SUBDOMAIN],
    stdout=None,
    stderr=None,
    start_new_session=True,
)
localtunnel_public_url = f"https://{LOCALTUNNEL_SUBDOMAIN}.loca.lt"
print(f"YOUR LOCALTUNNEL URL IS: {localtunnel_public_url}")
print("Keep this process and the Colab runtime alive while using cloud tasks.")


## 6. Background AI dependencies (run after the stable URL is available)
Run this cell after FastAPI and Localtunnel are already running. It may take several minutes, but it will not stop the detached server or tunnel.


In [ ]:
# Optional OpenSwarm reasoning dependencies plus raw Playwright.
!/content/venv/bin/pip uninstall -y browser-use || true
!/content/venv/bin/pip install playwright pydantic-settings langchain-openai langgraph
!/content/venv/bin/python -c "import playwright; print('Playwright successfully imported!')"
print("Playwright and optional OpenSwarm dependencies are ready in /content/venv.")


## Playwright-managed Chromium
The next cell installs Playwright’s own Chromium build; no system Chromium or snap wrapper is used.


In [ ]:
%%bash
set -u
export PLAYWRIGHT_BROWSERS_PATH="/content/.cache/ms-playwright"
export PLAYWRIGHT_HOST_PLATFORM_OVERRIDE="ubuntu24.04-x64"
mkdir -p "$PLAYWRIGHT_BROWSERS_PATH"
echo "Installing Playwright Chromium with a 300-second timeout..."
if timeout 300 /content/venv/bin/playwright install chromium; then
  echo "Playwright Chromium installation completed."
else
  status=$?
  echo "WARNING: Chromium installation failed or timed out (exit $status)."
  echo "The server remains online; browser tasks will report the missing runtime."
fi


In [ ]:
%%bash
set -e
export PLAYWRIGHT_BROWSERS_PATH="/content/.cache/ms-playwright"
/content/venv/bin/python - <<'PYCODE'
from pathlib import Path
import os
root = Path(os.environ["PLAYWRIGHT_BROWSERS_PATH"])
found = [p for pattern in ("chromium-*/chrome-linux/chrome", "chromium-*/chrome-linux64/chrome") for p in root.glob(pattern) if p.is_file() and os.access(p, os.X_OK)]
if not found:
    raise SystemExit(f"No executable Playwright Chromium found under {root}")
print(f"Playwright Chromium ready: {max(found, key=lambda p: p.stat().st_mtime)}")
PYCODE


## 8. Optional: test the raw Playwright scraper interactively
Run after the Playwright Chromium installation cell. It directly scrapes a public documentation page.


In [ ]:
%%bash
set +e
cd /content/usman-and-umar-app/cloud-backend
stdout_file=/tmp/playwright_test.stdout
stderr_file=/tmp/playwright_test.stderr
export PLAYWRIGHT_BROWSERS_PATH="/content/.cache/ms-playwright"
/content/venv/bin/python -c 'import asyncio; from browser_scraper import run_playwright_task; print(asyncio.run(run_playwright_task("https://docs.python.org/3/tutorial/", allowed_domains=["docs.python.org"])))' >"$stdout_file" 2>"$stderr_file"
status=$?
echo "===== Playwright stdout ====="
cat "$stdout_file"
echo "===== Playwright stderr ====="
cat "$stderr_file"
if [ "$status" -ne 0 ]; then
  echo "Playwright test failed with exit code $status."
  exit "$status"
fi
